# 🚀 Fine-Tuning XLM-RoBERTa for Pakistani NER (v4 Real-Only Package)
**Competition:** TechFest NER Challenge (20 Pakistani Entity Classes)
**Architecture:** `xlm-roberta-base`
**Data Split:** Real Collected Text Only (`v4/train_real.jsonl` - 5,282 sents, `v4/dev_real.jsonl` - 454 sents)
**PII Strategy:** Zero synthetic PII in training data (`PHONE_NUMBER` and `ID_NUMBER` handled deterministically via regex in hybrid mode)
**Augmentation:** Train-only 5% ALL-CAPS and 5% lowercase copies of real sentences (dev and benchmark kept 100% untouched)

> ⚠️ **CRITICAL WARNING ABOUT NESTED CLONE PATHS:**
> In Google Colab, executing duplicate git clone commands can create a nested subfolder:
> `/content/AI-Odyssey/ner-pk/AI-Odyssey/ner-pk`
> Always execute the directory check cell below to ensure `os.getcwd()` is strictly `/content/AI-Odyssey/ner-pk`.

In [ ]:
# 1. GPU Verification & Mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone Repository or Navigate with Nested Path Protection
import os

target_repo_dir = "/content/AI-Odyssey/ner-pk"
if not os.path.exists(target_repo_dir):
    %cd /content
    !git clone https://github.com/Muhammad-Hassan06/AI-Odyssey.git

# Set working directory safely and assert proper location
os.chdir(target_repo_dir)
print("Current Working Directory:", os.getcwd())
assert not os.path.exists("/content/AI-Odyssey/ner-pk/AI-Odyssey/ner-pk"), \
    "WARNING: Detected nested clone path! Remove duplicate before continuing."
assert os.path.exists("v4/train_real.jsonl"), f"Path error! v4/train_real.jsonl not found in {os.getcwd()}"
print("✔ Repository path and v4 data directory verified successfully!")

In [ ]:
# 3. Verify Environment & Library Versions (DO NOT DOWNGRADE)
import hashlib
import torch
import transformers
import seqeval
from pathlib import Path

print("--- Library Versions (Standard Modern Colab Runtime) ---")
print(f"PyTorch:      {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"Transformers: {transformers.__version__}")
print(f"Seqeval:      {seqeval.__version__}")

def get_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

print("\n--- v4 Real-Only Data Files Verification ---")
expected_files = ["v4/train_real.jsonl", "v4/dev_real.jsonl", "v4/benchmark_real.jsonl"]
for split in expected_files:
    p = Path(split)
    assert p.exists(), f"Missing {split}!"
    with open(p, "r", encoding="utf-8") as f:
        lines = sum(1 for line in f if line.strip())
    print(f"{split:<26}: {lines:>5} sentences | SHA256: {get_sha256(p)}")

In [ ]:
# 4. Validate Schema Prior to Training
# Strictly asserts 20 allowed classes, valid BIO tags, and no corrupted spans
!python scripts/validate_schema.py v4/train_real.jsonl
!python scripts/validate_schema.py v4/dev_real.jsonl
!python scripts/validate_schema.py v4/benchmark_real.jsonl

In [ ]:
# 5. Execute Fine-Tuning (8 Epochs, lr 2.5e-5, Batch 16, fp16, Class-Weighted Loss, Seed 42)
# Train-only casing augmentation adds 5% ALL-CAPS and 5% lowercase copies of real text
# --include_synthetic_pii is left OFF by default (handled by regex rules in hybrid mode)
# Checkpoint chosen strictly by best seqeval F1 on DEV
!python src/train.py \
    --model_name xlm-roberta-base \
    --data_dir v4 \
    --train_file v4/train_real.jsonl \
    --dev_file v4/dev_real.jsonl \
    --output_dir /content/drive/MyDrive/best_ner_model_v4 \
    --epochs 8 \
    --lr 2.5e-5 \
    --batch_size 16 \
    --max_length 256 \
    --fp16 \
    --use_class_weights \
    --augment_casing \
    --seed 42

In [ ]:
# 6. Verify Checkpoint Integrity
import os, json

model_dir = "/content/drive/MyDrive/best_ner_model_v4"
assert os.path.exists(os.path.join(model_dir, "model.safetensors")), "model.safetensors missing!"
assert os.path.exists(os.path.join(model_dir, "config.json")), "config.json missing!"

with open(os.path.join(model_dir, "config.json")) as f:
    cfg = json.load(f)
assert len(cfg["id2label"]) == 41, f"Expected 41 labels, got {len(cfg['id2label'])}"
print("✔ Checkpoint strictly verified: exactly 41 BIO labels matching 20 competition classes!")

In [ ]:
# 7. Model-Swap Decision Rule & Verification
"""
MODEL-SWAP DECISION RULE:
Swap models/final/ with this new checkpoint ONLY IF:
1. The new checkpoint beats the current checkpoint on v4/dev_real.jsonl:
   - Current Checkpoint Dev Real Baseline:
     * Micro-F1: 82.69%
     * Macro-F1: 60.06%
   - New checkpoint must exceed both Macro-F1 (>60.06%) AND Micro-F1 (>82.69%).
2. New checkpoint passes scripts/validate_schema.py without warnings or errors.
3. Human auditor explicitly reviews and approves the replacement.
"""
print("Model-swap criteria defined. To package for download:")
!zip -r /content/best_ner_model_v4.zip /content/drive/MyDrive/best_ner_model_v4
print("✔ Checkpoint archive ready: /content/best_ner_model_v4.zip")